In [57]:
#importing necessary libraries
import numpy as np 
import pandas as pd
import os
import re
import glob
from pathlib import Path

In [58]:
import sys
print(sys.version)


3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [59]:
#DATA_DIR = r"C:\Users\Samreet\Desktop\DataAnalysis_Revision\DA_Course\Python\PythonHackathon_Sep2026\Python_Hackathon_Sep_2026\cardiac_failure_Dataset"
df = pd.read_csv("data/raw/patient_precriptions.csv")
#csv_files = glob.glob(DATA_DIR + "/*.csv") #Retrieve the directory full path
#csv_files=os.listdir(DATA_DIR)            #Retrieve the file name from the directory
#csv_files                                  #show the list of all csv files along with their full path

**DATA CLEANING: PATIENT PRESCRIPTION CSV FILE**

In [60]:
#To read the csv file and saved it copy as df_clean (DEMOGRAPHY DATA FILE)
#df_clean = pd.read_csv(r"C:\Users\Samreet\Desktop\DataAnalysis_Revision\DA_Course\Python\PythonHackathon_Sep2026\Python_Hackathon_Sep_2026\cardiac_failure_Dataset\patient_precriptions.csv")
presc_clean=df.copy()

In [61]:
# Preview the content of the Data file
print(presc_clean.head())
print(presc_clean.shape)
print(presc_clean.columns)
print(presc_clean.tail())
print(len(presc_clean))


   inpatient_number                                         drug_name
0            857781                  sulfotanshinone sodium injection
1            857781                                 Furosemide tablet
2            857781                       Enoxaparin Sodium injection
3            857781  Meglumine Adenosine Cyclophosphate for injection
4            857781                              Furosemide injection
(15362, 2)
Index(['inpatient_number', 'drug_name'], dtype='object')
       inpatient_number                     drug_name
15357            791864  Valsartan Dispersible tablet
15358            791864                Digoxin tablet
15359            791864         Deslanoside injection
15360            791864           Milrinone injection
15361            791864          Furosemide injection
15362


In [62]:
presc_clean.columns.tolist()


['inpatient_number', 'drug_name']

In [63]:
# To Strip spaces & lowercase all drug names in order to be consistent 
presc_clean['drug_name'] = presc_clean['drug_name'].str.strip().str.lower()


In [64]:
print(presc_clean.head(10))

   inpatient_number                                         drug_name
0            857781                  sulfotanshinone sodium injection
1            857781                                 furosemide tablet
2            857781                       enoxaparin sodium injection
3            857781  meglumine adenosine cyclophosphate for injection
4            857781                              furosemide injection
5            857781                               milrinone injection
6            857781     metoprolol succinate sustained-release tablet
7            857781                             deslanoside injection
8            857781                                 torasemide tablet
9            857781                   benazepril hydrochloride tablet


In [65]:
# Checking data type of patients id's (column is inpatient_number)
column_dtype = presc_clean['inpatient_number'].dtype
print(f"The column data type is: {column_dtype}")

# Identify non-numeric ids 
non_numeric_mask = pd.to_numeric(presc_clean['inpatient_number'], errors='coerce').isna()
non_numeric_ids = presc_clean[non_numeric_mask]
if non_numeric_ids.empty:
    print("All patient IDs are completely numeric.")
else:
    print(f"Found {len(non_numeric_ids)} row(s) with non-numeric or missing patient IDs.\n")
    print(non_numeric_ids)

The column data type is: int64
All patient IDs are completely numeric.


In [66]:
#To Verify to find any duplicate records and print the number of duplicated rows if any
duplicate_count = presc_clean.duplicated().sum()
if duplicate_count > 0:
    print(f"Yes, the dataset contains {duplicate_count} duplicate rows.")
else:
    print("No duplicate rows found in the dataset.")


No duplicate rows found in the dataset.


In [67]:
# Another set of verification to find all duplicate records based on both patient ID and drug name in data file
duplicate_patients = presc_clean[presc_clean.duplicated(subset=['inpatient_number', 'drug_name'], keep=False)]

if not duplicate_patients.empty:
    # Count how many duplicate rows exist per patient ID
    duplicate_counts_per_patient = duplicate_patients.groupby('inpatient_number').size() // 2
    
    print(f"Found {duplicate_counts_per_patient.count()} unique patient ID(s) with duplicate prescriptions.\n")
    print("--- Patient IDs and their number of duplicate entries ---")
    print(duplicate_counts_per_patient.to_string())
else:
    print("No patient IDs have duplicate entries for the same drug.")

No patient IDs have duplicate entries for the same drug.


In [68]:
# Rename the column permanently
presc_clean = presc_clean.rename(columns={'inpatient_number': 'patient_id'})

In [69]:
print(presc_clean[['patient_id', 'drug_name']].head())

   patient_id                                         drug_name
0      857781                  sulfotanshinone sodium injection
1      857781                                 furosemide tablet
2      857781                       enoxaparin sodium injection
3      857781  meglumine adenosine cyclophosphate for injection
4      857781                              furosemide injection


**Combined the multiple drug name records for each patient into a single medication list so that all prescribed drugs are retained**
**Reason:maintaining one unique row per patient for merging with the other CSV files**

In [70]:
#prescription_summary = (presc_clean.groupby("patient_id")["drug_name"].apply(lambda x: "; ".join(x.dropna().astype(str).unique())) .reset_index() )


In [71]:
#print("Total rows:", len(prescription_summary)) 
#print("Unique patients:", prescription_summary["patient_id"].nunique()) 
#print("Duplicate patient IDs:", prescription_summary["patient_id"].duplicated().sum())


In [72]:
#presc_clean["prescription_count"] = ( presc_clean.groupby("patient_id")["drug_name"] .transform("nunique") )

In [81]:
# Group by patient_id to combine drug names and count total medications
presc_combined = presc_clean.groupby('patient_id').agg(
    drug_name=('drug_name', ', '.join),
    drug_count=('drug_name', 'count')
).reset_index()

print(presc_combined.head())


   patient_id                                          drug_name  drug_count
0      722128  furosemide tablet, spironolactone tablet, milr...           7
1      723327  clopidogrel hydrogen sulphate tablet, furosemi...          12
2      723617  heparin sodium injection, furosemide injection...           4
3      724385  furosemide tablet, furosemide injection, spiro...           6
4      725509  furosemide injection, shenfu injection, milrin...           9


**Saving the final version of all changes in to "patient_prescriptions_cleaned.csv"**

In [83]:
presc_combined.to_csv('data/cleaned/patient_prescriptions_cleaned.csv', index=False)
print("\nCleaned dataset exported successfully as 'patient_prescriptions_cleaned.csv'!")


Cleaned dataset exported successfully as 'patient_prescriptions_cleaned.csv'!
